# LSGAN — least-squares loss keeps the gradients flowing

> Tutorial pair for [`lsgan.py`](lsgan.py).

## 1. Intuition
With the usual sigmoid cross-entropy (BCE) loss, once a fake lands on the correct
side of the decision boundary the discriminator is "happy" and BCE **saturates**
— even if that fake is still far from the real data. **LSGAN** scores the
discriminator's raw output with a least-squares (L2) penalty, which keeps pushing
samples in proportion to how far they are from the target value. "Correct but far"
fakes still get a strong gradient pulling them toward the data.

## 2. Concept (the slide)
- **Discriminator** outputs a single *unbounded* value (no sigmoid).
- Targets use the $a$–$b$–$c$ coding: D pushes real $\to b$, fake $\to a$; G pushes
  fake $\to c$. The standard choice is $a=0,\ b=1,\ c=1$.
- Replacing BCE with L2 means the loss never flattens, so the generator keeps
  receiving useful gradients.

## 3. Math derivation — least squares and the Pearson $\chi^2$ divergence

LSGAN replaces the log-loss with squared error. With labels $a$ (fake), $b$
(real), $c$ (the value G wants its fakes to score), the objectives are
$$\min_D\;\tfrac12\,\mathbb E_{x\sim p_{\text{data}}}\big[(D(x)-b)^2\big]
 +\tfrac12\,\mathbb E_{z\sim p_z}\big[(D(G(z))-a)^2\big],$$
$$\min_G\;\tfrac12\,\mathbb E_{z\sim p_z}\big[(D(G(z))-c)^2\big].$$

**Optimal discriminator.** For fixed $G$, minimizing pointwise over $D(x)$ gives
$$D^\star(x)=\frac{b\,p_{\text{data}}(x)+a\,p_g(x)}{p_{\text{data}}(x)+p_g(x)}.$$

**What G minimizes.** Substitute $D^\star$ into G's objective. Choosing labels with
$b-c=1$ and $b-a=2$ (e.g. $a=-1,b=1,c=0$, or equivalently the demo's $a=0,b=1,c=1$
up to a shift) yields, after algebra,
$$2\,C(G)=\int\frac{\big((b-c)(p_{\text{data}}+p_g)-(b-a)p_g\big)^2}
 {p_{\text{data}}+p_g}\,dx
 =\chi^2_{\text{Pearson}}\big(p_{\text{data}}+p_g\,\big\Vert\,2p_g\big),$$
the **Pearson $\chi^2$ divergence** between $p_{\text{data}}+p_g$ and $2p_g$,
minimized iff $p_g=p_{\text{data}}$.

**Why this differs from vanilla GAN.** The vanilla GAN minimizes the
Jensen–Shannon divergence through a *sigmoid* output: for a confidently-classified
fake, $\partial_{\text{logit}}\,\mathrm{BCE}\to 0$, so its gradient **vanishes**.
The L2 loss has gradient $\propto (D(G(z))-c)$, which grows *linearly* with the
error and never saturates — distant fakes are penalized hardest, exactly where a
GAN most needs signal. The cost is the JS$\to\chi^2$ change of divergence.

## 4. Generator / key component

In [ ]:
# ===== actual implementation from lsgan.py =====
from __future__ import annotations

import numpy as np

import torch

import torch.nn as nn

SEED = 0

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def make_ring(n: int = 2000, k: int = 8, r: float = 2.0, seed: int = SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    ang = 2 * np.pi * rng.integers(0, k, n) / k
    centers = np.c_[r * np.cos(ang), r * np.sin(ang)]
    return (centers + 0.1 * rng.normal(size=(n, 2))).astype(np.float32)

class Generator(nn.Module):
    def __init__(self, noise_dim: int = 8, data_dim: int = 2, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, data_dim))

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)

class Discriminator(nn.Module):
    """Outputs a single *unbounded* real value (no sigmoid): LSGAN scores it with L2."""

    def __init__(self, data_dim: int = 2, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

## 5. Trainer / losses

In [ ]:
# ===== actual implementation from lsgan.py =====
class LSGANTorch:
    """Least-squares GAN. Labels a (fake target), b (real target), c (G target)."""

    def __init__(self, noise_dim: int = 8, data_dim: int = 2, lr: float = 2e-4,
                 a: float = 0.0, b: float = 1.0, c: float = 1.0):
        torch.manual_seed(SEED)
        self.dev = get_device()
        self.noise_dim = noise_dim
        self.a, self.b, self.c = a, b, c
        self.G = Generator(noise_dim, data_dim).to(self.dev)
        self.D = Discriminator(data_dim).to(self.dev)
        self.optG = torch.optim.Adam(self.G.parameters(), lr=lr, betas=(0.5, 0.999))
        self.optD = torch.optim.Adam(self.D.parameters(), lr=lr, betas=(0.5, 0.999))

    @staticmethod
    def _mse(out: torch.Tensor, target: float) -> torch.Tensor:
        return 0.5 * ((out - target) ** 2).mean()

    def fit(self, real: np.ndarray, steps: int = 1500, batch: int = 128):
        real = torch.as_tensor(real, dtype=torch.float32, device=self.dev)
        self.d_hist, self.g_hist = [], []
        for _ in range(steps):
            idx = torch.randint(0, len(real), (batch,), device=self.dev)
            x = real[idx]
            z = torch.randn(batch, self.noise_dim, device=self.dev)
            # --- D step: push D(real)->b, D(fake)->a ---
            fake = self.G(z).detach()
            lossD = self._mse(self.D(x), self.b) + self._mse(self.D(fake), self.a)
            self.optD.zero_grad(); lossD.backward(); self.optD.step()
            # --- G step: push D(fake)->c (so fakes look "real" to D) ---
            z = torch.randn(batch, self.noise_dim, device=self.dev)
            lossG = self._mse(self.D(self.G(z)), self.c)
            self.optG.zero_grad(); lossG.backward(); self.optG.step()
            self.d_hist.append(lossD.item()); self.g_hist.append(lossG.item())
        return self

    @torch.no_grad()
    def generate(self, n: int) -> np.ndarray:
        z = torch.randn(n, self.noise_dim, device=self.dev)
        return self.G(z).cpu().numpy()

def _coverage(fake: np.ndarray, k: int = 8) -> int:
    modes = np.arctan2(fake[:, 1], fake[:, 0])
    return len(np.unique(np.round(modes / (2 * np.pi / k)).astype(int) % k))

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    real = make_ring(2000)
    print(f"real mean={real.mean(0).round(2)} std={real.std(0).round(2)}")

    gan = LSGANTorch().fit(real, steps=1500, batch=128)
    fake = gan.generate(800)
    d0, d1 = np.mean(gan.d_hist[:100]), np.mean(gan.d_hist[-100:])
    print(f"D (L2) loss trend: {d0:.3f} -> {d1:.3f}")
    print(f"G (L2) loss trend: {np.mean(gan.g_hist[:100]):.3f} -> "
          f"{np.mean(gan.g_hist[-100:]):.3f}")
    print(f"covered {_coverage(fake)}/8 modes; fake mean={fake.mean(0).round(2)} "
          f"std={fake.std(0).round(2)}")

## 6. Train

In [ ]:
demo()

## 7. Visualization

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import lsgan as M

real = M.make_ring(2000)
gan = M.LSGANTorch().fit(real, steps=1500, batch=128)
fake = gan.generate(1000)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(real[:, 0], real[:, 1], s=6, alpha=.3, label="real")
ax[0].scatter(fake[:, 0], fake[:, 1], s=6, alpha=.5, color="r", label="fake")
ax[0].set_title("Real vs generated (LSGAN)"); ax[0].legend(); ax[0].set_aspect("equal")
ax[1].plot(gan.d_hist, label="D (L2) loss", alpha=.7)
ax[1].plot(gan.g_hist, label="G (L2) loss", alpha=.7)
ax[1].set_xlabel("step"); ax[1].set_title("Least-squares losses"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- Swapping BCE for L2 turns the JS-divergence game into a **Pearson $\chi^2$**
  game whose gradient grows with the error instead of saturating.
- Practical win: distant "already classified" fakes still get pulled toward the
  data, reducing vanishing-gradient stalls.
- Pitfalls: the unbounded D output can blow up without care; label coding matters
  ($b-c=1,\ b-a=2$ for the clean $\chi^2$ interpretation); still not immune to
  mode collapse.